In [6]:
import os
import ast
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, MultiLabelBinarizer
from xgboost import XGBClassifier


pd.set_option("display.max_columns", None)

In [3]:
# Games
GAMES_PATH = r"D:\lol draft analyzer - datascientest\200kmatchs\draft_simple.csv"
df_games = pd.read_csv(GAMES_PATH)

# Champions
CHAMPS_PATH = r"D:\lol draft analyzer\part 2\data champions\data champions avec winrate global\champions_15.1.1_15.24.1(1).csv"
df_champs = pd.read_csv(CHAMPS_PATH, encoding="utf-8")


C:\Users\samue\AppData\Local\Temp\ipykernel_139216\3029721481.py:3: DtypeWarning: Columns (10,11,12,13,14,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  df_games = pd.read_csv(GAMES_PATH)


In [8]:
from sklearn.preprocessing import TargetEncoder

In [ ]:
# Encodage des valeurs catégorielles
# - One hot
# - Ordinal encoder (exemple good lane bad lane)
# - Target encoder (intéressant sur les valeurs à forte cardinalité)
#
# Garder les dates
# - Calculer des winrates glissantes (par joueur et par date)
# - Idem par joueur et par game

In [1]:
df_games.head(10)

NameError: name 'df_games' is not defined

In [5]:
df_games.columns.tolist()

['match_id',
 'serveur',
 'patch',
 'elo',
 'blue_side_win',
 'blue_top_champion',
 'blue_jungle_champion',
 'blue_mid_champion',
 'blue_adc_champion',
 'blue_support_champion',
 'blue_top_puuid',
 'blue_jungle_puuid',
 'blue_mid_puuid',
 'blue_adc_puuid',
 'blue_support_puuid',
 'red_top_champion',
 'red_jungle_champion',
 'red_mid_champion',
 'red_adc_champion',
 'red_support_champion',
 'red_top_puuid',
 'red_jungle_puuid',
 'red_mid_puuid',
 'red_adc_puuid',
 'red_support_puuid',
 'blue_ban_1',
 'blue_ban_2',
 'blue_ban_3',
 'blue_ban_4',
 'blue_ban_5',
 'red_ban_1',
 'red_ban_2',
 'red_ban_3',
 'red_ban_4',
 'red_ban_5']

In [7]:
df_champs.head(10)

,patch,id,key,name,title,partype,tags,lore,blurb,allytips,enemytips,skins,skins_count,info_attack,info_defense,info_magic,info_difficulty,stats_hp,stats_hpperlevel,stats_mp,stats_mpperlevel,stats_movespeed,stats_armor,stats_armorperlevel,stats_spellblock,stats_spellblockperlevel,stats_attackrange,stats_hpregen,stats_hpregenperlevel,stats_mpregen,stats_mpregenperlevel,stats_crit,stats_critperlevel,stats_attackdamage,stats_attackdamageperlevel,stats_attackspeedperlevel,stats_attackspeed,passive_name,passive_description,passive_image,spell_q_id,spell_q_name,spell_q_description,spell_q_tooltip,spell_q_leveltip,spell_q_maxrank,spell_q_cooldown,spell_q_cooldownBurn,spell_q_cost,spell_q_costBurn,spell_q_datavalues,spell_q_effect,spell_q_effectBurn,spell_q_vars,spell_q_costType,spell_q_maxammo,spell_q_range,spell_q_rangeBurn,spell_q_image,spell_q_resource,spell_w_id,spell_w_name,spell_w_description,spell_w_tooltip,spell_w_leveltip,spell_w_maxrank,spell_w_cooldown,spell_w_cooldownBurn,spell_w_cost,spell_w_costBurn,spell_w_datavalues,spell_w_effect,spell_w_effectBurn,spell_w_vars,spell_w_costType,spell_w_maxammo,spell_w_range,spell_w_rangeBurn,spell_w_image,spell_w_resource,spell_e_id,spell_e_name,spell_e_description,spell_e_tooltip,spell_e_leveltip,spell_e_maxrank,spell_e_cooldown,spell_e_cooldownBurn,spell_e_cost,spell_e_costBurn,spell_e_datavalues,spell_e_effect,spell_e_effectBurn,spell_e_vars,spell_e_costType,spell_e_maxammo,spell_e_range,spell_e_rangeBurn,spell_e_image,spell_e_resource,spell_r_id,spell_r_name,spell_r_description,spell_r_tooltip,spell_r_leveltip,spell_r_maxrank,spell_r_cooldown,spell_r_cooldownBurn,spell_r_cost,spell_r_costBurn,spell_r_datavalues,spell_r_effect,spell_r_effectBurn,spell_r_vars,spell_r_costType,spell_r_maxammo,spell_r_range,spell_r_rangeBurn,spell_r_image,spell_r_resource,winrate
0,15.1.1,Aatrox,266,Aatrox,Épée des Darkin,Puits de sang,"[""Fighter""]","Autrefois, Aatrox et ses frères étaient honoré...","Autrefois, Aatrox et ses frères étaient honoré...","[""Utilisez Ruée obscure tout en lançant Épée d...","[""Les attaques d'Aatrox sont prévisibles. Prof...","[{""id"": ""266000"", ""num"": 0, ""name"": ""default"",...",13,8,4,3,4,650,114,0,0.0,345,38,4.8,32,2.05,175,3.00,0.50,0.0,0.00,0,0,60,5.00,2.500,0.651,Posture du massacreur,"Régulièrement, la prochaine attaque de base d'...","{""full"": ""Aatrox_Passive.png"", ""sprite"": ""pass...",AatroxQ,Épée des Darkin,"Aatrox abat son épée devant lui, infligeant de...","Aatrox abat son épée, infligeant <physicalDama...","{""label"": [""Délai de récupération"", ""Dégâts"", ...",5,"[14, 12, 10, 8, 6]",14/12/10/8/6,"[0, 0, 0, 0, 0]",0,{},"[null, [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],Pas de coût,-1,"[25000, 25000, 25000, 25000, 25000]",25000,"{""full"": ""AatroxQ.png"", ""sprite"": ""spell0.png""...",Pas de coût,AatroxW,Chaînes infernales,"Aatrox frappe le sol, blessant le premier enne...","Aatrox lance une chaîne, <status>ralentissant<...","{""label"": [""Délai de récupération"", ""Dégâts"", ...",5,"[20, 18, 16, 14, 12]",20/18/16/14/12,"[0, 0, 0, 0, 0]",0,{},"[null, [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],Pas de coût,-1,"[825, 825, 825, 825, 825]",825,"{""full"": ""AatroxW.png"", ""sprite"": ""spell0.png""...",Pas de coût,AatroxE,Ruée obscure,"Passivement, Aatrox se soigne quand il blesse ...",<spellPassive>Passive :</spellPassive> Aatrox ...,"{""label"": [""Délai de récupération""], ""effect"":...",5,"[9, 8, 7, 6, 5]",9/8/7/6/5,"[0, 0, 0, 0, 0]",0,{},"[null, [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],Pas de coût,-1,"[25000, 25000, 25000, 25000, 25000]",25000,"{""full"": ""AatroxE.png"", ""sprite"": ""spell0.png""...",Pas de coût,AatroxR,Fossoyeur des mondes,"Aatrox libère sa forme démoniaque, effrayant l...","Aatrox révèle sa vraie forme dé

In [10]:
df_games.head()

,match_id,serveur,patch,elo,blue_side_win,blue_top_champion,blue_jungle_champion,blue_mid_champion,blue_adc_champion,blue_support_champion,...,blue_ban_1,blue_ban_2,blue_ban_3,blue_ban_4,blue_ban_5,red_ban_1,red_ban_2,red_ban_3,red_ban_4,red_ban_5
0,EUW1_7517331727,euw1,15.17.1,HIGH_ELO,False,Sett,Talon,Diana,Jinx,Rell,...,Lulu,Fiora,Rengar,Vex,Poppy,Dr. Mundo,Garen,Milio,Zoe,Draven
1,EUW1_7509322616,euw1,15.17.1,HIGH_ELO,False,Mordekaiser,Qiyana,Yasuo,Corki,Nami,...,Yunara,Pyke,Gwen,Kayle,Smolder,Master Yi,Pantheon,Twitch,Draven,Fiora
2,EUW1_7509243675,euw1,15.17.1,HIGH_ELO,False,Smolder,Kindred,Galio,Jinx,Milio,...,Dr. Mundo,Rengar,Fiora,Master Yi,Yasuo,Master Yi,Twitch,Akali,Draven,Aatrox
3,EUW1_7509193063,euw1,15.17.1,HIGH_ELO,False,Warwick,Nidalee,Akali,Yunara,Rakan,...,Sivir,Qiyana,Naafiri,Draven,Twitch,Fiddlesticks,Blitzcrank,Mel,Volibear,Darius
4,EUW1_7508909082,euw1,15.17.1,HIGH_ELO,False,Aurora,XinZhao,Twitch,Yunara,Rakan,...,Draven,Kassadin,Shaco,Ahri,Senna,Nocturne,Evelynn,Galio,Yasuo,Riven


In [6]:
df_champs.columns.tolist()

['patch',
 'id',
 'key',
 'name',
 'title',
 'partype',
 'tags',
 'lore',
 'blurb',
 'allytips',
 'enemytips',
 'skins',
 'skins_count',
 'info_attack',
 'info_defense',
 'info_magic',
 'info_difficulty',
 'stats_hp',
 'stats_hpperlevel',
 'stats_mp',
 'stats_mpperlevel',
 'stats_movespeed',
 'stats_armor',
 'stats_armorperlevel',
 'stats_spellblock',
 'stats_spellblockperlevel',
 'stats_attackrange',
 'stats_hpregen',
 'stats_hpregenperlevel',
 'stats_mpregen',
 'stats_mpregenperlevel',
 'stats_crit',
 'stats_critperlevel',
 'stats_attackdamage',
 'stats_attackdamageperlevel',
 'stats_attackspeedperlevel',
 'stats_attackspeed',
 'passive_name',
 'passive_description',
 'passive_image',
 'spell_q_id',
 'spell_q_name',
 'spell_q_description',
 'spell_q_tooltip',
 'spell_q_leveltip',
 'spell_q_maxrank',
 'spell_q_cooldown',
 'spell_q_cooldownBurn',
 'spell_q_cost',
 'spell_q_costBurn',
 'spell_q_datavalues',
 'spell_q_effect',
 'spell_q_effectBurn',
 'spell_q_vars',
 'spell_q_costType',


In [11]:
df_champs.head()

,patch,id,key,name,title,partype,tags,lore,blurb,allytips,...,spell_r_effect,spell_r_effectBurn,spell_r_vars,spell_r_costType,spell_r_maxammo,spell_r_range,spell_r_rangeBurn,spell_r_image,spell_r_resource,winrate
0,15.1.1,Aatrox,266,Aatrox,Épée des Darkin,Puits de sang,"[""Fighter""]","Autrefois, Aatrox et ses frères étaient honoré...","Autrefois, Aatrox et ses frères étaient honoré...","[""Utilisez Ruée obscure tout en lançant Épée d...",...,"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],Pas de coût,-1,"[25000, 25000, 25000]",25000,"{""full"": ""AatroxR.png"", ""sprite"": ""spell0.png""...",Pas de coût,0.501607
1,15.1.1,Ahri,103,Ahri,Renard à neuf queues,Mana,"[""Mage"", ""Assassin""]","Connectée à la magie du royaume spirituel, Ahr...","Connectée à la magie du royaume spirituel, Ahr...","[""Utilisez Charme pour préparer vos combos, ce...",...,"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],{{ abilityresourcename }},-1,"[450, 450, 450]",450,"{""full"": ""AhriR.png"", ""sprite"": ""spell0.png"", ...",{{ cost }} {{ abilityresourcename }},0.516904
2,15.1.1,Akali,84,Akali,Assassin rebelle,Énergie,"[""Assassin""]",Ayant abandonné l'Ordre Kinkou et le titre de ...,Ayant abandonné l'Ordre Kinkou et le titre de ...,"[""Akali peut facilement tuer les champions fra...",...,"[null, [0, 0, 0], [0, 0, 0], [1, 1, 1], [0, 0,...","[null, ""0"", ""0"", ""1"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],Pas de coût,-1,"[675, 675, 675]",675,"{""full"": ""AkaliR.png"", ""sprite"": ""spell0.png"",...",Pas de coût,0.500777
3,15.1.1,Akshan,166,Akshan,Sentinelle rebelle,Mana,"[""Marksman"", ""Assassin""]","Se jouant du danger, Akshan combat le mal sans...","Se jouant du danger, Akshan combat le mal sans...","[""Se jouant du danger, Akshan combat le mal sa...",...,"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],{{ abilityresourcename }},-1,"[2500, 2500, 2500]",2500,"{""full"": ""AkshanR.png"", ""sprite"": ""spell0.png""...",{{ cost }} {{ abilityresourcename }},0.507645
4,15.1.1,Alistar,12,Alistar,Minotaure,Mana,"[""Tank"", ""Support""]",Alistar est un guerrier redoutable cherchant à...,Alistar est un guerrier redoutable cherchant à...,"[""Atomisation peut vous aider à mieux vous pla...",...,"[null, [0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0,...","[null, ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"", ""0"",...",[],{{ abilityresourcename }},-1,"[1, 1, 1]",1,"{""full"": ""FerociousHowl.png"", ""sprite"": ""spell...",{{ cost }} {{ abilityresourcename }},0.490315


In [8]:
df_games["patch"] = (
    df_games["patch"]
    .str.split(".")
    .str[:2]
    .str.join(".")
    + ".1"
)


In [18]:
champ_cols = [
    "blue_top_champion", "blue_jungle_champion", "blue_mid_champion", "blue_adc_champion", "blue_support_champion",
    "red_top_champion", "red_jungle_champion", "red_mid_champion", "red_adc_champion", "red_support_champion"
]

stat_cols = [
    'partype', 'tags',
    'info_attack', 'info_defense', 'info_magic', 'info_difficulty',
    'stats_hp', 'stats_hpperlevel',
    'stats_mp', 'stats_mpperlevel',
    'stats_movespeed', 'stats_armor', 'stats_armorperlevel',
    'stats_spellblock', 'stats_spellblockperlevel',
    'stats_attackrange',
    'stats_attackdamage', 'stats_attackdamageperlevel',
    'stats_attackspeed', 'stats_attackspeedperlevel', 'winrate'
]


In [27]:
df_champs_small = df_champs[["patch", "name"] + stat_cols]

df_final = df_games.copy()

for col in champ_cols:
    df_final = df_final.merge(
        df_champs_small,
        left_on=["patch", col],
        right_on=["patch", "name"],
        how="left"
    )
    df_final = df_final.rename(
        columns={s: f"{col}_{s}" for s in stat_cols}
    ).drop(columns=["name"])


In [28]:
df_final = df_final.drop(
    columns=champ_cols + ["patch", "match_id"],
    errors="ignore"
)


In [29]:
df_final.columns.tolist()

['serveur',
 'elo',
 'blue_side_win',
 'blue_top_puuid',
 'blue_jungle_puuid',
 'blue_mid_puuid',
 'blue_adc_puuid',
 'blue_support_puuid',
 'red_top_puuid',
 'red_jungle_puuid',
 'red_mid_puuid',
 'red_adc_puuid',
 'red_support_puuid',
 'blue_ban_1',
 'blue_ban_2',
 'blue_ban_3',
 'blue_ban_4',
 'blue_ban_5',
 'red_ban_1',
 'red_ban_2',
 'red_ban_3',
 'red_ban_4',
 'red_ban_5',
 'blue_top_champion_partype',
 'blue_top_champion_tags',
 'blue_top_champion_info_attack',
 'blue_top_champion_info_defense',
 'blue_top_champion_info_magic',
 'blue_top_champion_info_difficulty',
 'blue_top_champion_stats_hp',
 'blue_top_champion_stats_hpperlevel',
 'blue_top_champion_stats_mp',
 'blue_top_champion_stats_mpperlevel',
 'blue_top_champion_stats_movespeed',
 'blue_top_champion_stats_armor',
 'blue_top_champion_stats_armorperlevel',
 'blue_top_champion_stats_spellblock',
 'blue_top_champion_stats_spellblockperlevel',
 'blue_top_champion_stats_attackrange',
 'blue_top_champion_stats_attackdamage',


In [30]:
cols_to_drop = [
 'blue_top_puuid',
 'blue_jungle_puuid',
 'blue_mid_puuid',
 'blue_adc_puuid',
 'blue_support_puuid',
 'red_top_puuid',
 'red_jungle_puuid',
 'red_mid_puuid',
 'red_adc_puuid',
 'red_support_puuid',
 'blue_ban_1',
 'blue_ban_2',
 'blue_ban_3',
 'blue_ban_4',
 'blue_ban_5',
 'red_ban_1',
 'red_ban_2',
 'red_ban_3',
 'red_ban_4',
 'red_ban_5']
df_final = df_final.drop(columns=cols_to_drop, errors="ignore")

In [31]:
partype_cols = [c for c in df_final.columns if c.endswith("_partype")]
df_final[partype_cols] = df_final[partype_cols].fillna("Unknown")

df_partype_encoded = pd.get_dummies(
    df_final[partype_cols],
    prefix=partype_cols
)


In [32]:
tag_cols = [c for c in df_final.columns if c.endswith("_tags")]

for col in tag_cols:
    df_final[col] = df_final[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )

mlb = MultiLabelBinarizer()
tags_encoded = []

for col in tag_cols:
    encoded = pd.DataFrame(
        mlb.fit_transform(df_final[col]),
        columns=[f"{col}_{c}" for c in mlb.classes_],
        index=df_final.index
    )
    tags_encoded.append(encoded)

df_tags_encoded = pd.concat(tags_encoded, axis=1)


In [34]:
num_cols = df_final.select_dtypes(include="number").columns.tolist()
print(num_cols)
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df_final[num_cols]),
    columns=num_cols,
    index=df_final.index
)


['blue_top_champion_info_attack', 'blue_top_champion_info_defense', 'blue_top_champion_info_magic', 'blue_top_champion_info_difficulty', 'blue_top_champion_stats_hp', 'blue_top_champion_stats_hpperlevel', 'blue_top_champion_stats_mp', 'blue_top_champion_stats_mpperlevel', 'blue_top_champion_stats_movespeed', 'blue_top_champion_stats_armor', 'blue_top_champion_stats_armorperlevel', 'blue_top_champion_stats_spellblock', 'blue_top_champion_stats_spellblockperlevel', 'blue_top_champion_stats_attackrange', 'blue_top_champion_stats_attackdamage', 'blue_top_champion_stats_attackdamageperlevel', 'blue_top_champion_stats_attackspeed', 'blue_top_champion_stats_attackspeedperlevel', 'blue_top_champion_winrate', 'blue_jungle_champion_info_attack', 'blue_jungle_champion_info_defense', 'blue_jungle_champion_info_magic', 'blue_jungle_champion_info_difficulty', 'blue_jungle_champion_stats_hp', 'blue_jungle_champion_stats_hpperlevel', 'blue_jungle_champion_stats_mp', 'blue_jungle_champion_stats_mpperle

In [36]:
X = pd.concat(
    [df_scaled, df_partype_encoded, df_tags_encoded],
    axis=1
)

y = df_final['blue_side_win']


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [38]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [39]:
#tester accuracy
accuracy = model.score(X_test, y_test)
print(f"Accuracy: {accuracy}")

Accuracy: 0.5322064056939502


In [40]:
os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/xgb_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(X.columns.tolist(), "models/feature_columns.pkl")

print("✅ Modèle et artefacts sauvegardés")


✅ Modèle et artefacts sauvegardés


In [ ]:
# ON introduit un threshold pour ne parier que sur les games sur lesquelles on est confiant sur l'issue

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score

# Probabilités sur le jeu de test
y_proba = model.predict_proba(X_test)[:, 1]  # proba classe 1
y_true = y_test.values

results = []

for x in range(0, 21):  # x de 0 à 20
    margin = x / 100  # ex: x=10 -> 0.10
    
    lower = 0.5 - margin
    upper = 0.5 + margin

    # Indices des prédictions "confiantes"
    confident_mask = (y_proba <= lower) | (y_proba >= upper)

    # Si aucune prédiction retenue, on skip
    if confident_mask.sum() == 0:
        continue

    # Prédictions binaires uniquement sur les confiantes
    y_pred_confident = (y_proba[confident_mask] >= 0.5).astype(int)
    y_true_confident = y_true[confident_mask]

    accuracy = accuracy_score(y_true_confident, y_pred_confident)
    coverage = confident_mask.mean()  # % de matchs prédits

    results.append({
        "x (%)": x,
        "threshold_low": round(lower, 2),
        "threshold_high": round(upper, 2),
        "accuracy": accuracy,
        "coverage": coverage,
        "n_predictions": confident_mask.sum()
    })

results_df = pd.DataFrame(results)

results_df


,x (%),threshold_low,threshold_high,accuracy,coverage,n_predictions
0,0,0.50,0.50,0.532206,1.000000,56200
1,1,0.49,0.51,0.540267,0.833843,46862
2,2,0.48,0.52,0.545727,0.676299,38008
3,3,0.47,0.53,0.551457,0.533559,29986
4,4,0.46,0.54,0.557619,0.409181,22996
5,5,0.45,0.55,0.564313,0.306139,17205
6,6,0.44,0.56,0.571234,0.222313,12494
7,7,0.43,0.57,0.582152,0.158114,8886
8,8,0.42,0.58,0.582356,0.110730,6223
9,9,0.41,0.59,0.593542,0.074947,4212


In [45]:
pip install scikit-optimize xgboost


     -------------------------------------- 107.8/107.8 KB 6.5 MB/s eta 0:00:00
     -------------------------------------- 158.8/158.8 KB 9.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [ ]:
# #Bayesian search pour optimiser le modele
# from xgboost import XGBClassifier
# from skopt import BayesSearchCV
# from skopt.space import Real, Integer, Categorical
# from sklearn.metrics import accuracy_score
# import numpy as np

# xgb = XGBClassifier(
#     objective="binary:logistic",
#     random_state=42,
#     n_jobs=-1,
#     use_label_encoder=False
# )

# search_spaces = {
#     "n_estimators": Integer(200, 1000),
#     "max_depth": Integer(3, 8),
#     "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
#     "subsample": Real(0.6, 1.0),
#     "colsample_bytree": Real(0.6, 1.0),
#     "min_child_weight": Integer(1, 10),
#     "gamma": Real(0, 5),

#     # ⚡ Nouveaux paramètres
#     "eval_metric": Categorical(["logloss", "auc", "error"])
# }

# bayes_search = BayesSearchCV(
#     estimator=xgb,
#     search_spaces=search_spaces,
#     n_iter=30,
#     scoring="accuracy",
#     cv=3,
#     verbose=2,
#     random_state=42,
#     n_jobs=-1
# )

# bayes_search.fit(X_train, y_train)

# best_model = bayes_search.best_estimator_

# print("Meilleurs paramètres :")
# print(bayes_search.best_params_)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


KeyboardInterrupt: 

In [ ]:
#Bayesian search pour optimiser le modele
from xgboost import XGBClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.metrics import accuracy_score
import numpy as np


xgb = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False
)

search_spaces = {
    "n_estimators": Integer(200, 1000),
    "max_depth": Integer(3, 8),
    "learning_rate": Real(0.01, 0.3, prior="log-uniform"),
    "subsample": Real(0.6, 1.0),
    "colsample_bytree": Real(0.6, 1.0),
    "min_child_weight": Integer(1, 10),
    "gamma": Real(0, 5),

    # ⚡ Nouveaux paramètres
    "eval_metric": Categorical(["logloss", "auc", "error"])
}

bayes_search = BayesSearchCV(
    estimator=xgb,
    search_spaces=search_spaces,
    n_iter=150,
    scoring="accuracy",
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

bayes_search.fit(X_train, y_train)

best_model = bayes_search.best_estimator_

print("Meilleurs paramètres :")
print(bayes_search.best_params_)


Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fi

c:\Users\samue\Documents\datascientest-lol-draft_analyzer\venv\lib\site-packages\xgboost\core.py:158: UserWarning: [07:27:13] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Meilleurs paramètres :
OrderedDict([('colsample_bytree', 1.0), ('eval_metric', 'auc'), ('gamma', 0.0), ('learning_rate', 0.027809848029046458), ('max_depth', 3), ('min_child_weight', 1), ('n_estimators', 1000), ('subsample', 0.6)])


In [53]:
y_pred = best_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy classique : {acc:.4f}")


Accuracy classique : 0.5351


In [54]:
y_proba = best_model.predict_proba(X_test)[:, 1]

results = []

for x in range(0, 21):
    low = 0.5 - x / 100
    high = 0.5 + x / 100

    mask = (y_proba <= low) | (y_proba >= high)

    if mask.sum() == 0:
        continue

    y_pred_conf = (y_proba[mask] >= 0.5).astype(int)
    y_true_conf = y_test.values[mask]

    acc_conf = accuracy_score(y_true_conf, y_pred_conf)

    results.append({
        "x_%": x,
        "accuracy": acc_conf,
        "coverage": mask.mean(),
        "n_samples": mask.sum()
    })

df_results = pd.DataFrame(results)
df_results


,x_%,accuracy,coverage,n_samples
0,0,0.535142,1.000000,56200
1,1,0.541183,0.820267,46099
2,2,0.548698,0.651299,36603
3,3,0.555216,0.500142,28108
4,4,0.563048,0.371263,20865
5,5,0.570874,0.265747,14935
6,6,0.582355,0.184947,10394
7,7,0.599455,0.124075,6973
8,8,0.606954,0.080854,4544
9,9,0.610059,0.051655,2903


In [ ]:
os.makedirs("models-150-iter", exist_ok=True)

joblib.dump(model, "models-150-iter/xgb_model.pkl")
joblib.dump(scaler, "models-150-iter/scaler.pkl")
joblib.dump(X.columns.tolist(), "models-150-iter/feature_columns.pkl")

print("✅ Modèle et artefacts sauvegardés")

✅ Modèle et artefacts sauvegardés


: 